# WSI with lazyslide

0. Colab setup  (run on Google Colab, skip it when running locally)
1. **Setup** (imports + settings)
2. **Preprocess** ( detect tissue, cut into tiles)
3. **Features** (foundation-model features + spatial domains)
4. **Zero-shot**  (training-free tumor score)
5. **Validate** [ROC-AUC vs a ground-truth annotation (skips if none)]
6. **Spatial** (neighbour-graph visualization)
7. **Batch** (run every slide in `slides/` and collect a results table)


## 0. Colab setup (skip it locally)


In [ ]:

!apt-get -qq install -y openslide-tools
!pip install -q "lazyslide[all,models]" leidenalg igraph

#   small file -> from google.colab import files; files.upload()
#   large file -> from google.colab import drive; drive.mount('/content/drive')
#                 SLIDE = "/content/drive/MyDrive/path/to/your_slide.tif"

## 1. Setup

In [ ]:
# ==== EDIT THESE ====
SLIDE = "slides/tumor_084.tif"   
MODEL = "plip"                   # foundation model: "plip" or "conch"
TILE_PX = 256                    # tile size in pixels
TILE_MPP = 0.5                   # microns/pixel (~20x)
RESOLUTION = 0.2                 # spatial-domain clustering resolution

# zero-shot text prompts (wording matters a lot; edit for non-lymph-node tissue!)
CLASSES = {
    "tumor":  "metastatic carcinoma, malignant tumor cells, atypical epithelial cells",
    "normal": "normal lymph node tissue, small mature lymphocytes",
}
VALIDATE_CLASS = "tumor"         # which class to validate / overlay

# validation ground truth (optional):
ANNOTATION = None               # explicit .xml/.geojson path, or None to auto-detect
POSITIVE_LABELS = ["tumor", "metastas"]  # label(s) that mark tumor; [] = every region
# ====================

%matplotlib inline
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import lazyslide as zs

# use mixed precision (amp) only on GPU; on a CPU-only machine use plain precision
AMP = bool(torch.cuda.is_available()
           or (hasattr(torch.backends, "mps") and torch.backends.mps.is_available()))


here = Path.cwd()
REPO = next((p for p in [here, *here.parents] if (p / "slides").exists()), here)
os.chdir(REPO)

slide = Path(SLIDE)
# fail fast with a clear message instead of a confusing error a few cells later
if not slide.exists():
    raise FileNotFoundError(f"slide not found: {slide}  (looked under {Path.cwd()})")

labels = list(CLASSES.keys())
outputs = Path("outputs"); outputs.mkdir(exist_ok=True)
store = outputs / f"{slide.stem}.zarr"   # results persist here between cells

print(f"repo   : {REPO}")
print(f"slide  : {slide}   (exists: {slide.exists()})")
print(f"store  : {store}")
print(f"model  : {MODEL}")
print(f"device : {'GPU (amp on)' if AMP else 'CPU (amp off)'}")
print("ready — run the stage cells below in order.")

In [ ]:

# Used by both the Validate cell (section 5) and the batch cell (section 7).

import xml.etree.ElementTree as ET, json
import geopandas as gpd
from shapely.geometry import Polygon, shape
from shapely.ops import unary_union
from sklearn.metrics import roc_auc_score

def positive_match(label):
    """Does this annotation label count as the positive (tumor) class?"""
    if not POSITIVE_LABELS:                 # empty list -> every region is positive
        return True
    label = (label or "").lower()
    return any(label.startswith(p.lower()) for p in POSITIVE_LABELS)

def load_annotation(path):
    """Return (unioned positive geometry, n_regions) in level-0 pixel coords.
    Understands .xml (ASAP/CAMELYON16) and .geojson/.json (QuPath)."""
    path = Path(path)
    if path.suffix.lower() == ".xml":            # ASAP / CAMELYON16
        root = ET.parse(path).getroot(); polys = []
        for a in root.findall(".//Annotation"):
            if positive_match(a.get("PartOfGroup")):
                pts = [(float(c.get("X")), float(c.get("Y"))) for c in a.findall(".//Coordinate")]
                if len(pts) >= 3: polys.append(Polygon(pts))
    else:                                        # QuPath-style GeoJSON
        data = json.loads(path.read_text())
        feats = data.get("features", data if isinstance(data, list) else [data])
        polys = []
        for f in feats:
            props = f.get("properties", {}) or {}
            cls = (props.get("classification") or {}).get("name") or props.get("name")
            if not positive_match(cls): continue
            geom = shape(f["geometry"]); polys.extend(getattr(geom, "geoms", [geom]))
    return unary_union([p.buffer(0) for p in polys]), len(polys)

def find_annotation(slide):
    """Same-basename annotation next to the slide (.xml/.geojson/.json), or None."""
    for ext in (".xml", ".geojson", ".json"):
        cand = slide.with_suffix(ext)
        if cand.exists(): return cand
    return None

print("annotation helpers ready: positive_match, load_annotation, find_annotation")

## 2. Preprocess (detect tissue, cut into tiles)

In [ ]:
# open the slide, backed by a .zarr store so results persist on wsi.write()
wsi = zs.open_wsi(str(slide), backed_file=str(store))
mpp = wsi.properties.mpp
mpp_str = f"{mpp:.3f}" if mpp is not None else "UNKNOWN"
print(f"opened {slide.stem}: {wsi.properties.shape}  mpp={mpp_str}")
if mpp is None:
    print("WARNING: this slide reports no mpp (microns/pixel) metadata. Tiling at a")
    print("micron target (TILE_MPP) may fail — a slide with mpp metadata is needed.")

zs.pp.find_tissues(wsi)                                  # -> wsi.shapes["tissues"]
print(f"tissue pieces: {len(wsi.shapes['tissues'])}")

zs.pp.tile_tissues(wsi, tile_px=TILE_PX, mpp=TILE_MPP)   # -> wsi.shapes["tiles"]
print(f"tiles: {len(wsi.shapes['tiles'])}")

zs.pl.tissue(wsi); plt.show()
zs.pl.tiles(wsi, linewidth=0.5); plt.show()

wsi.write(overwrite=True)
print(f"saved -> {store}")

## 3. Features (foundation-model features + spatial domains)

In [ ]:
wsi = zs.open_wsi(str(slide), backed_file=str(store))
print(f"tiles: {len(wsi['tiles'])}")

# extract tile features only if not already done 
# set FORCE_FEATURES = True in the Setup cell to recompute from scratch
if f"{MODEL}_tiles" not in wsi.tables or globals().get("FORCE_FEATURES", False):
    zs.tl.feature_extraction(wsi, MODEL, amp=AMP)      # one vector per tile -> "<model>_tiles"
else:
    print(f"reusing existing {MODEL} features")


# (re-run this cell after changing RESOLUTION to re-cluster without re-extracting)
zs.tl.feature_aggregation(wsi, feature_key=MODEL)       # slide-level vector
zs.pp.tile_graph(wsi)                                   # neighbour graph -> "tiles_graph"
zs.tl.spatial_features(wsi, MODEL)                      # smooth features over neighbours
zs.tl.spatial_domain(wsi, feature_key=MODEL, layer="spatial_features",
                     resolution=RESOLUTION)             # cluster -> "domain" column
print(f"spatial domains: {wsi['tiles']['domain'].nunique()}")

zs.pl.tiles(wsi, color="domain", alpha=0.5); plt.show()

wsi.write(overwrite=True)
print(f"features ({MODEL}) + domains saved -> {store}")

## 4. Zero-shot (training-free tumor score)

In [ ]:
wsi = zs.open_wsi(str(slide), backed_file=str(store))
print(f"zero-shot model: {MODEL}")

# reuse tile features from step 3 if present, else extract now
if f"{MODEL}_tiles" not in wsi.tables:
    zs.tl.feature_extraction(wsi, MODEL, amp=AMP)

# embed the class prompts with the model's text encoder
embeds = zs.tl.text_embedding([CLASSES[k] for k in labels], model=MODEL)
embeds.index = labels

# per-tile probability for each class -> table "zeroshot"
zs.tl.text_image_similarity(wsi, embeds, model=MODEL, feature_key=f"{MODEL}_tiles",
                            key_added="zeroshot", softmax=True)

for lab in labels:
    zs.pl.tiles(wsi, feature_key="zeroshot", color=lab, style="heatmap")
    plt.title(f"zero-shot: {lab}"); plt.show()

wsi.write(overwrite=True)
print(f"zero-shot scores saved -> table 'zeroshot' in {store}")

## 5. Validate (ROC-AUC vs ground truth)

Reads a `.xml` (ASAP/CAMELYON16) or `.geojson` (QuPath) annotation next to the slide,
or the `ANNOTATION` path you set. Skips automatically if none is found.

In [ ]:

ann = Path(ANNOTATION) if ANNOTATION else find_annotation(slide)

if ann is None or not ann.exists():
    print("no annotation found — skipping validation")
else:
    wsi = zs.open_wsi(str(slide), backed_file=str(store))
    tiles = wsi["tiles"].copy()

    # align zero-shot scores to tiles by tile_id
    zt = wsi.tables["zeroshot"]
    scores = zt.obs[["tile_id"]].copy()
    scores["score"] = np.asarray(zt[:, VALIDATE_CLASS].X).ravel()
    tiles = tiles.merge(scores, on="tile_id", how="left")

    tumor, n = load_annotation(ann)
    print(f"loaded {n} positive region(s) from {ann.name}")

    # label each tile by whether its centroid falls inside the tumor
    centroids = gpd.GeoDataFrame(geometry=tiles.geometry.centroid, crs=tiles.crs)
    tiles["is_tumor"] = centroids.geometry.within(tumor).astype(int)

    ok = tiles["score"].notna()
    if tiles.loc[ok, "is_tumor"].nunique() < 2:
        # tumor-free slide, or tumor covers all tissue -> ROC-AUC is undefined
        print("only one class present (all tiles inside OR all outside the tumor) —")
        print("ROC-AUC is undefined for this slide; showing the heatmap only.")
        auc = None
    else:
        auc = roc_auc_score(tiles.loc[ok, "is_tumor"], tiles.loc[ok, "score"])
        mean_in = tiles.loc[ok & (tiles["is_tumor"] == 1), "score"].mean()
        mean_out = tiles.loc[ok & (tiles["is_tumor"] == 0), "score"].mean()
        print(f"mean '{VALIDATE_CLASS}' INSIDE tumor : {mean_in:.4f}")
        print(f"mean '{VALIDATE_CLASS}' OUTSIDE      : {mean_out:.4f}")
        print(f"ROC-AUC vs ground truth        : {auc:.3f}   (0.5 = no signal, 1.0 = perfect)")

    # overlay the ground-truth outline (red) on the heatmap (shown either way)
    zs.pl.tiles(wsi, feature_key="zeroshot", color=VALIDATE_CLASS, style="heatmap")
    gpd.GeoSeries(tumor).boundary.plot(ax=plt.gca(), color="red", linewidth=1.2)
    title = f"zero-shot {VALIDATE_CLASS} vs ground truth (red = tumor)"
    if auc is not None:
        title += f"\nAUC={auc:.3f}"
    plt.gca().set_title(title)
    plt.show()

## 6. Spatial (neighbour-graph visualization)

In [ ]:
from matplotlib.collections import LineCollection

wsi = zs.open_wsi(str(slide), backed_file=str(store))
tiles = wsi["tiles"]

# this plot needs the tile graph + "domain" labels from the Features cell (section 3),

if "tiles_graph" not in wsi.tables or "domain" not in tiles.columns:
    raise RuntimeError("run the Features cell (section 3) in this session first — "
                       "it builds the tile graph and 'domain' labels this plot needs.")

# node positions = tile centroids (level-0 pixel coordinates)
cent = tiles.geometry.centroid
xy = np.column_stack([cent.x.to_numpy(), cent.y.to_numpy()])

# edges = neighbour graph; upper triangle so each edge is drawn once
A = wsi.tables["tiles_graph"].obsp["spatial_connectivities"].tocoo()
keep = A.row < A.col
segments = np.stack([xy[A.row[keep]], xy[A.col[keep]]], axis=1)
domain = tiles["domain"].astype(int).to_numpy()

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 10))

# left: whole slide, nodes coloured by domain (edges faint)
ax0.add_collection(LineCollection(segments, colors="0.85", linewidths=0.2, zorder=1))
sc = ax0.scatter(xy[:, 0], xy[:, 1], c=domain, cmap="tab10", s=5, zorder=2)
ax0.set_title(f"{slide.stem}: {domain.max() + 1} spatial domains")
plt.colorbar(sc, ax=ax0, label="spatial domain", shrink=0.5,
             ticks=range(int(domain.max()) + 1))

# right: a zoom window where the neighbour edges are actually visible
cx, cy = xy.mean(axis=0)
win = 6000  # level-0 pixels (~10 tiles across)
in_win = (np.abs(xy[:, 0] - cx) < win) & (np.abs(xy[:, 1] - cy) < win)
edge_in = in_win[A.row[keep]] & in_win[A.col[keep]]
ax1.add_collection(LineCollection(segments[edge_in], colors="0.5", linewidths=0.6, zorder=1))
ax1.scatter(xy[in_win, 0], xy[in_win, 1], c=domain[in_win], cmap="tab10", s=45, zorder=2)
ax1.set_xlim(cx - win, cx + win); ax1.set_ylim(cy - win, cy + win)
ax1.set_title(f"zoom: neighbour graph ({int(edge_in.sum())} edges shown)")

for ax in (ax0, ax1):
    ax.set_aspect("equal")
    ax.invert_yaxis()                   # image coords: y increases downward
    ax.axis("off")

plt.show()
print(f"nodes: {len(xy)}   edges: {len(segments)}   domains: {domain.max() + 1}")

## 7. Batch (run every slide in `slides/`)

Runs the whole pipeline (preprocess → features → zero-shot → validate) on every
slide in `slides/`, saves each slide's plots to `outputs/`, and collects one results
table + a per-slide AUC bar chart. The spatial-graph viz (section 6) is skipped here.

Slides without an annotation still run; their AUC is
just left blank. Re-running skips slides whose scores already exist (pass
`force=True` to `run_slide` to recompute).

In [ ]:
# batch runner 
def save_current(png_name):
    plt.savefig(outputs / png_name, dpi=150, bbox_inches="tight"); plt.close()

def run_slide(slide, force=False):
    """Full pipeline for ONE slide. Returns a dict of results; saves PNGs to outputs/.
    Skips the heavy stages if the slide's store already has zero-shot scores
    (pass force=True to recompute)."""
    store = outputs / f"{slide.stem}.zarr"
    wsi = zs.open_wsi(str(slide), backed_file=str(store))

    n_domains = None
    if force or "zeroshot" not in wsi.tables:
        # --- preprocess ---
        zs.pp.find_tissues(wsi)
        zs.pp.tile_tissues(wsi, tile_px=TILE_PX, mpp=TILE_MPP)
        # --- features + spatial domains ---
        zs.tl.feature_extraction(wsi, MODEL, amp=AMP)
        zs.tl.feature_aggregation(wsi, feature_key=MODEL)
        zs.pp.tile_graph(wsi)
        zs.tl.spatial_features(wsi, MODEL)
        zs.tl.spatial_domain(wsi, feature_key=MODEL, layer="spatial_features", resolution=RESOLUTION)
        n_domains = int(wsi["tiles"]["domain"].nunique())   # capture while it's live
        # --- zero-shot ---
        embeds = zs.tl.text_embedding([CLASSES[k] for k in labels], model=MODEL); embeds.index = labels
        zs.tl.text_image_similarity(wsi, embeds, model=MODEL, feature_key=f"{MODEL}_tiles",
                                    key_added="zeroshot", softmax=True)
        wsi.write(overwrite=True)
        # save plots to files (not inline — many slides would flood the notebook)
        zs.pl.tiles(wsi, color="domain", alpha=0.5); save_current(f"{slide.stem}_domains.png")
        for lab in labels:
            zs.pl.tiles(wsi, feature_key="zeroshot", color=lab, style="heatmap")
            save_current(f"{slide.stem}_zeroshot_{lab}.png")

    # --- validate (only if an annotation exists) ---
    auc, n_pos, ann = None, 0, find_annotation(slide)
    if ann is not None:
        tiles = wsi["tiles"].copy()
        zt = wsi.tables["zeroshot"]
        sc = zt.obs[["tile_id"]].copy(); sc["score"] = np.asarray(zt[:, VALIDATE_CLASS].X).ravel()
        tiles = tiles.merge(sc, on="tile_id", how="left")
        tumor, n_pos = load_annotation(ann)
        cent = gpd.GeoDataFrame(geometry=tiles.geometry.centroid, crs=tiles.crs)
        tiles["is_tumor"] = cent.geometry.within(tumor).astype(int)
        ok = tiles["score"].notna()
        if tiles.loc[ok, "is_tumor"].nunique() == 2:   # need both classes for AUC
            auc = roc_auc_score(tiles.loc[ok, "is_tumor"], tiles.loc[ok, "score"])
        zs.pl.tiles(wsi, feature_key="zeroshot", color=VALIDATE_CLASS, style="heatmap")
        gpd.GeoSeries(tumor).boundary.plot(ax=plt.gca(), color="red", linewidth=1.2)
        plt.gca().set_title(f"{slide.stem}  AUC={auc:.3f}" if auc is not None else slide.stem)
        save_current(f"{slide.stem}_validation.png")

    return {"slide": slide.stem, "n_tiles": len(wsi["tiles"]), "n_domains": n_domains,
            "annotation": ann.name if ann else None, "n_pos_regions": n_pos, "auc": auc}

print("run_slide() defined — run the batch cell below.")

In [ ]:
import pandas as pd
import time

# pick up every slide in slides/ (common WSI extensions)
exts = ("*.tif", "*.tiff", "*.svs", "*.ndpi")
slide_paths = sorted(p for e in exts for p in Path("slides").glob(e))
print(f"found {len(slide_paths)} slide(s):", [p.name for p in slide_paths])

rows = []
for i, s in enumerate(slide_paths, 1):
    t0 = time.time()
    print(f"[{i}/{len(slide_paths)}] {s.stem} ...", end=" ", flush=True)
    try:
        r = run_slide(s)                                   # one bad slide won't stop the rest
        r["seconds"] = round(time.time() - t0, 1)
        rows.append(r)
        print(f"done  auc={r['auc']}  ({r['seconds']}s)")
    except Exception as e:
        rows.append({"slide": s.stem, "auc": None, "error": repr(e)})
        print(f"FAILED: {e}")

summary = pd.DataFrame(rows)
summary.to_csv(outputs / "cohort_summary.csv", index=False)
print("\nsaved -> outputs/cohort_summary.csv")

valid = summary["auc"].dropna()
if len(valid):
    print(f"cohort mean AUC: {valid.mean():.3f}   (over {len(valid)} annotated slide(s))")

# display() is a built-in in Jupyter kernels — no import needed
display(summary)

# per-slide AUC bar chart (annotated slides only)
annotated = summary.dropna(subset=["auc"])
if len(annotated):
    ax = annotated.plot.bar(x="slide", y="auc", legend=False, figsize=(10, 4))
    ax.axhline(0.5, color="grey", ls="--", label="chance")
    ax.set_ylabel("ROC-AUC"); ax.set_ylim(0, 1)
    ax.set_title("Per-slide zero-shot AUC"); plt.tight_layout(); plt.show()